[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q2/02_hub_identification.ipynb)

# R1-Q2 Week 3 — Identify hub genes in stress-relevant modules

This notebook identifies hubs in each per-tissue network, restricted to the modules that are most relevant to stress response.

By the end of this notebook you will have:

- A defended list of stress-relevant modules for shoot and root, produced by your pre-committed criterion.
- Per-tissue hub gene tables ranked by kME (`shoot_hubs.parquet`, `root_hubs.parquet`).
- A cross-tissue overlap picture showing which hubs appear in both shoot and root — the strongest tissue-independent candidates.
- A `hub_summary.json` ready as Week 3 draft-presentation input.

In [ ]:
# There are three patterns for installing irilab2026 from GitHub, depending on your needs. 

# The first is for a fresh, complete runtime. This installs irilab2026 and every dependency it declares. 
# It skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

!pip install git+https://github.com/geraldmc/irilab2026.git@main
 
# The second is for code iteration only. It reinstalls irilab2026 itself, ignoring its dep list entirely. 
# The dep tree is left exactly as it is. This is ideal for iterating on irilab2026 itself, when you know 
# the deps are already satisfied and don't want to waste time reinstalling them.
# 
# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

# The third is for developers who want to work with the code directly and see changes reflected 
# immediately without reinstalling.

#!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main
#!pip install git+https://github.com/geraldmc/irilab2026.git@main

## 1) Load Notebook 1's networks and confirm structure

Notebook 1 built per-tissue co-expression networks with PyWGCNA and saved them as Python pickle files. This section loads both `shoot.p` and `root.p`, confirms the basic shape of each (samples, genes, modules), and verifies that the sample metadata attached in Notebook 1's Section 5.1 survived the pickle round-trip.

The metadata check matters because the first analytical step — finding stress-relevant modules — needs the per-sample stress labels to correlate against. Catching a missing-metadata problem here saves a confusing failure several cells later.

In [ ]:
# Mount Drive, set up irilab2026, point to the R1-Q2 output directory,
# and load the pre-commit you wrote in Notebook 0.
import pickle

shoot_path = OUTPUT_DIR / "shoot.p"
root_path = OUTPUT_DIR / "root.p"

try:
    with open(shoot_path, "rb") as f:
        shoot_wgcna = pickle.load(f)
    with open(root_path, "rb") as f:
        root_wgcna = pickle.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find a Notebook 1 output.\n"
        f"  Looked for: {shoot_path}\n"
        f"              {root_path}\n\n"
        f"Run `01_wgcna.ipynb` to completion first — its final section "
        f"saves both pickles to this location.\n"
    ) from None

print(f"Loaded shoot.p  ({shoot_path.stat().st_size:>10,} bytes)")
print(f"Loaded root.p   ({root_path.stat().st_size:>10,} bytes)")

In [ ]:
# Structure check — confirm shape, module count, and metadata for both networks.
# PyWGCNA's AnnData orientation (after the orientation fix in N1) is:
#   datExpr.obs = samples (124 rows; carries the metadata attached in N1's S5.1)
#   datExpr.var = genes  (~5,687 rows; carries module assignments)

expected_metadata = {"stress", "tissue"}

for label, wgcna in [("shoot", shoot_wgcna), ("root", root_wgcna)]:
    n_samples, n_genes = wgcna.datExpr.shape
    n_modules = wgcna.datExpr.var["moduleColors"].nunique()

    obs_cols = set(wgcna.datExpr.obs.columns)
    missing = expected_metadata - obs_cols

    print(f"{label}:")
    print(f"  shape:        {n_samples} samples × {n_genes} genes")
    print(f"  modules:      {n_modules}")
    print(f"  obs columns:  {sorted(obs_cols)}")
    if missing:
        print(f"  WARNING: missing expected metadata: {missing}")
    print()

Both networks loaded with the shape Notebook 1 reported (124 samples × ~5,700 genes, 9 modules per tissue) and the sample metadata is on the `obs` axis. The next section applies the module-relevance pre-commit to find which modules are stress-relevant per tissue.